# Lab | Chains in LangChain

## Outline

* LLMChain
* Sequential Chains
  * SimpleSequentialChain
  * SequentialChain
* Router Chain

In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')
HUGGINGFACEHUB_API_TOKEN = os.getenv('HUGGINGFACEHUB_API_TOKEN')

In [3]:
import pandas as pd
df = pd.read_csv('data/Data.csv')

In [4]:
df.head()

,Product,Review
0,Queen Size Sheet Set,I ordered a king size set. My only criticism w...
1,Waterproof Phone Pouch,"I loved the waterproof sac, although the openi..."
2,Luxury Air Mattress,This mattress had a small hole in the top of i...
3,Pillows Insert,This is the best throw pillow fillers on Amazo...
4,Milk Frother Handheld\n,I loved this product. But they only seem to l...


## LLMChain

In [7]:
from langchain_openai import ChatOpenAI        # ✅ 1.0.3
from langchain_core.prompts import ChatPromptTemplate  # ✅ From langchain-core 1.0.5
from langchain_classic.chains import LLMChain  # ✅ Legacy (install if missing)

In [8]:
#Replace None by your own value and justify
llm = ChatOpenAI(temperature=0.2)


In [9]:
prompt = ChatPromptTemplate.from_template(
    "Write a short, vivid product description for a {product}. "
    "Include 3 key features and a one-sentence tagline."
)

In [10]:

chain = LLMChain(llm=llm, prompt=prompt)

/tmp/ipykernel_21562/546483037.py:1: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use `RunnableSequence, e.g., `prompt | llm`` instead.
  chain = LLMChain(llm=llm, prompt=prompt)


In [11]:
product = "portable espresso maker"
chain.run(product)

/tmp/ipykernel_21562/2126017940.py:2: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain-classic 0.1.0 and will be removed in 1.0. Use `invoke` instead.
  chain.run(product)


'Introducing the ultimate on-the-go espresso experience with our portable espresso maker! \n\nKey features:\n1. Compact and lightweight design for easy transport anywhere.\n2. No electricity or batteries required - simply add hot water and your favorite coffee grounds for a perfect espresso shot.\n3. Durable construction for long-lasting use, perfect for camping, hiking, or traveling.\n\nTagline: Brew your perfect espresso wherever you go with our portable espresso maker!'

## SimpleSequentialChain

In [12]:
from langchain_classic.chains import SimpleSequentialChain  

In [13]:
llm = ChatOpenAI(temperature=0.9)

# prompt template 1
first_prompt = ChatPromptTemplate.from_template(
    "You are a creative marketing assistant. Write a catchy product name and a 1-sentence concept for a {product}. "
    "Keep it short and punchy."
)

# Chain 1
chain_one = LLMChain(llm=llm, prompt=first_prompt)



In [14]:
# prompt template 2
second_prompt = ChatPromptTemplate.from_template(
    "Take the following product concept and write a 4-bullet ad. "
    "Each bullet should be 6-10 words. End with a tagline. \n\n"
    "Concept: {concept}"
)
# chain 2
chain_two = LLMChain(llm=llm, prompt=second_prompt)

In [15]:
overall_simple_chain = SimpleSequentialChain(chains=[chain_one, chain_two],
                                             verbose=True
                                            )

In [16]:
overall_simple_chain.run(product)



> Entering new SimpleSequentialChain chain...
Name: Espresso on the Go

Concept: Get your caffeine fix anytime, anywhere with this sleek and compact portable espresso maker.
- Brew fresh espresso in minutes on the go
- Lightweight and portable design fits in your bag
- Easy to use and clean for busy lifestyles
- Enjoy a delicious shot of espresso anywhere you are

Espresso on the Go: Caffeine at your fingertips.

> Finished chain.


'- Brew fresh espresso in minutes on the go\n- Lightweight and portable design fits in your bag\n- Easy to use and clean for busy lifestyles\n- Enjoy a delicious shot of espresso anywhere you are\n\nEspresso on the Go: Caffeine at your fingertips.'

**Repeat the above twice for different products**

## SequentialChain

In [17]:
from langchain_classic.chains import SequentialChain  

In [18]:
llm = ChatOpenAI(temperature=0.9)

# Chain 1: translate the review to English
first_prompt = ChatPromptTemplate.from_template(
    "Translate the following customer review to English. "
    "If it's already in English, just return it as-is.\n\n"
    "Review: {Review}"
)

chain_one = LLMChain(llm=llm, prompt=first_prompt,
                     output_key="English_Review"
                    )

In [19]:
# Chain 2: summarize the English review in 1 sentence
second_prompt = ChatPromptTemplate.from_template(
    "Summarize the following product review in 1 sentence.\n\n"
    "Review: {English_Review}"
)

chain_two = LLMChain(llm=llm, prompt=second_prompt,
                     output_key="summary"
                    )

In [20]:
# Chain 3: detect what language the original review was written in
third_prompt = ChatPromptTemplate.from_template(
    "What language was the following review originally written in? "
    "Just return the language name, nothing else.\n\n"
    "Review: {Review}"
)
# chain 3: input= Review and output= language
chain_three = LLMChain(llm=llm, prompt=third_prompt,
                       output_key="language"
                      )



In [21]:
fourth_prompt = ChatPromptTemplate.from_template(
    "You are a customer service agent. Write a polite follow-up message "
    "in {language} to a customer who left this review summary: {summary}. "
    "Thank them and address any concerns briefly."
)
chain_four = LLMChain(llm=llm, prompt=fourth_prompt,
                      output_key="followup_message"
                     )



In [22]:
overall_chain = SequentialChain(
    chains=[chain_one, chain_two, chain_three, chain_four],
    input_variables=["Review"],
    output_variables=["English_Review", "summary", "followup_message"],
    verbose=True
)

In [23]:
#First run 
review = df.Review[5]
overall_chain(review)



> Entering new SequentialChain chain...


/tmp/ipykernel_21562/1532879411.py:3: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain-classic 0.1.0 and will be removed in 1.0. Use `invoke` instead.
  overall_chain(review)



> Finished chain.


{'Review': "Je trouve le goût médiocre. La mousse ne tient pas, c'est bizarre. J'achète les mêmes dans le commerce et le goût est bien meilleur...\nVieux lot ou contrefaçon !?",
 'English_Review': "Review: I find the taste mediocre. The foam doesn't hold, it's weird. I buy the same ones in stores and the taste is much better... Old batch or counterfeit!?",
 'summary': 'The reviewer is disappointed with the taste and foam quality of the product, suggesting that it may be an old batch or counterfeit compared to the ones purchased in stores.',
 'followup_message': "Cher client,\n\nMerci d'avoir pris le temps de laisser votre avis. Nous sommes désolés d'apprendre que vous n'êtes pas entièrement satisfait de la qualité du produit que vous avez reçu. Nous prenons vos préoccupations très au sérieux et nous allons enquêter sur la question pour nous assurer que nos produits sont toujours frais et authentiques.\n\nNous vous remercions de nous avoir informés de votre expérience et nous vous prion

**Repeat the above twice for different products or reviews**

In [24]:
#Second run - trying with a different review from the dataset
review2 = df.Review[2]
overall_chain(review2)



> Entering new SequentialChain chain...

> Finished chain.


{'Review': "This mattress had a small hole in the top of it (took forever to find where it was), and the patches that they provide did not work, maybe because it's the top of the mattress where it's kind of like fabric and a patch won't stick. Maybe I got unlucky with a defective mattress, but where's quality assurance for this company? That flat out should not happen. Emphasis on flat. Cause that's what the mattress was. Seriously horrible experience, ruined my friend's stay with me. Then they make you ship it back instead of just providing a refund, which is also super annoying to pack up an air mattress and take it to the UPS store. This company is the worst, and this mattress is the worst.",
 'English_Review': "Translated Review: This mattress had a small hole in the top of it (took forever to find where it was), and the patches that they provide did not work, maybe because it's the top of the mattress where it's kind of like fabric and a patch won't stick. Maybe I got unlucky with

In [25]:
# Third run - one more review to see how it handles different writing styles
review3 = df.Review[3]
overall_chain(review3)



> Entering new SequentialChain chain...

> Finished chain.


{'Review': 'This is the best throw pillow fillers on Amazon. I’ve tried several others, and they’re all cheap and flat no matter how much fluffing you do. Once you toss these in the dryer after you remove them from the vacuum sealed shipping material, they fluff up great',
 'English_Review': 'Review: This is the best throw pillow fillers on Amazon. I’ve tried several others, and they’re all cheap and flat no matter how much fluffing you do. Once you toss these in the dryer after you remove them from the vacuum sealed shipping material, they fluff up great',
 'summary': 'These throw pillow fillers are the best on Amazon, staying fluffy even after being removed from vacuum sealed packaging.',
 'followup_message': 'Dear [Customer],\n\nThank you so much for your kind words and for taking the time to share your positive experience with our throw pillow fillers. We are thrilled to hear that they have exceeded your expectations and maintained their fluffiness even after being removed from the

## Router Chain

In [26]:
physics_template = """You are a very smart physics professor. \
You are great at answering questions about physics in a concise\
and easy to understand manner. \
When you don't know the answer to a question you admit\
that you don't know.

Here is a question:
{input}"""


math_template = """You are a very good mathematician. \
You are great at answering math questions. \
You are so good because you are able to break down \
hard problems into their component parts, 
answer the component parts, and then put them together\
to answer the broader question.

Here is a question:
{input}"""

history_template = """You are a very good historian. \
You have an excellent knowledge of and understanding of people,\
events and contexts from a range of historical periods. \
You have the ability to think, reflect, debate, discuss and \
evaluate the past. You have a respect for historical evidence\
and the ability to make use of it to support your explanations \
and judgements.

Here is a question:
{input}"""


computerscience_template = """ You are a successful computer scientist.\
You have a passion for creativity, collaboration,\
forward-thinking, confidence, strong problem-solving capabilities,\
understanding of theories and algorithms, and excellent communication \
skills. You are great at answering coding questions. \
You are so good because you know how to solve a problem by \
describing the solution in imperative steps \
that a machine can easily interpret and you know how to \
choose a solution that has a good balance between \
time complexity and space complexity. 

Here is a question:
{input}"""

biology_template = """You are an excellent biologist. \
You have a deep understanding of living organisms, \
from the molecular and cellular level to entire ecosystems. \
You are skilled at observing patterns in nature, analyzing biological data, \
and explaining complex processes like evolution, genetics, physiology, and ecology. \
You can clearly communicate how life functions and adapts, \
and you make connections between different biological concepts \
to answer challenging questions.

Here is a question:
{input}"""

In [27]:
prompt_infos = [
    {
        "name": "physics", 
        "description": "Good for answering questions about physics", 
        "prompt_template": physics_template
    },
    {
        "name": "math", 
        "description": "Good for answering math questions", 
        "prompt_template": math_template
    },
    {
        "name": "History", 
        "description": "Good for answering history questions", 
        "prompt_template": history_template
    },
    {
        "name": "computer science", 
        "description": "Good for answering computer science questions", 
        "prompt_template": computerscience_template
    },
    {
        "name": "biology",
        "description": "Good for answering biology questions",
        "prompt_template": biology_template
    }
]

In [28]:
from langchain_classic.chains.router import MultiPromptChain
from langchain_classic.chains.router.llm_router import LLMRouterChain, RouterOutputParser
from langchain_core.prompts import PromptTemplate  

In [29]:
llm = ChatOpenAI(temperature=0)

In [30]:
destination_chains = {}
for p_info in prompt_infos:
    name = p_info["name"]
    prompt_template = p_info["prompt_template"]
    prompt = ChatPromptTemplate.from_template(template=prompt_template)
    chain = LLMChain(llm=llm, prompt=prompt)
    destination_chains[name] = chain  
    
destinations = [f"{p['name']}: {p['description']}" for p in prompt_infos]
destinations_str = "\n".join(destinations)

In [31]:
default_prompt = ChatPromptTemplate.from_template("{input}")
default_chain = LLMChain(llm=llm, prompt=default_prompt)

In [32]:
MULTI_PROMPT_ROUTER_TEMPLATE = """Given a raw text input to a \
language model select the model prompt best suited for the input. \
You will be given the names of the available prompts and a \
description of what the prompt is best suited for. \
You may also revise the original input if you think that revising\
it will ultimately lead to a better response from the language model.

<< FORMATTING >>
Return a markdown code snippet with a JSON object formatted to look like:
```json
{{{{
    "destination": string \ name of the prompt to use or "DEFAULT"
    "next_inputs": string \ a potentially modified version of the original input
}}}}
```

REMEMBER: "destination" MUST be one of the candidate prompt \
names specified below OR it can be "DEFAULT" if the input is not\
well suited for any of the candidate prompts.
REMEMBER: "next_inputs" can just be the original input \
if you don't think any modifications are needed.

<< CANDIDATE PROMPTS >>
{destinations}

<< INPUT >>
{{input}}

<< OUTPUT (remember to include the ```json)>>"""

In [33]:
router_template = MULTI_PROMPT_ROUTER_TEMPLATE.format(
    destinations=destinations_str
)
router_prompt = PromptTemplate(
    template=router_template,
    input_variables=["input"],
    output_parser=RouterOutputParser(),
)

router_chain = LLMRouterChain.from_llm(llm, router_prompt)

In [34]:
chain = MultiPromptChain(router_chain=router_chain, 
                         destination_chains=destination_chains, 
                         default_chain=default_chain, verbose=True
                        )

/tmp/ipykernel_21562/3038952769.py:1: LangChainDeprecationWarning: Please see migration guide here for recommended implementation: https://python.langchain.com/docs/versions/migrating_chains/multi_prompt_chain/
  chain = MultiPromptChain(router_chain=router_chain,


In [35]:
chain.run("What is black body radiation?")



> Entering new MultiPromptChain chain...
physics: {'input': 'What is black body radiation?'}
> Finished chain.


"Black body radiation is the electromagnetic radiation emitted by a perfect absorber and emitter of radiation, known as a black body. A black body absorbs all radiation that falls on it and emits radiation across the entire electromagnetic spectrum. The spectrum of black body radiation is continuous and depends only on the temperature of the black body. This phenomenon is described by Planck's law, which states that the intensity of radiation emitted by a black body at a given wavelength is proportional to the temperature of the body and the wavelength raised to the fifth power."

In [36]:
chain.run("what is 2 + 2")



> Entering new MultiPromptChain chain...
math: {'input': 'what is 2 + 2'}
> Finished chain.


'The answer to 2 + 2 is 4.'

In [37]:
chain.run("Why does every cell in our body contain DNA?")



> Entering new MultiPromptChain chain...
biology: {'input': 'Why does every cell in our body contain DNA?'}
> Finished chain.


'Every cell in our body contains DNA because DNA is the genetic material that carries the instructions for the development, functioning, and reproduction of all living organisms. DNA contains the information needed to build and maintain an organism, including the proteins that make up our cells and tissues. \n\nHaving DNA in every cell ensures that each cell has the necessary genetic information to carry out its specific functions and to replicate itself accurately during cell division. This ensures that the genetic information is passed on to the next generation of cells. \n\nAdditionally, DNA is constantly being used by cells to carry out processes such as protein synthesis, cell division, and repair. Having DNA in every cell allows for the coordination of these processes and ensures that the organism functions properly as a whole. \n\nIn summary, every cell in our body contains DNA because it is essential for the proper functioning and development of all living organisms.'

**Repeat the above at least once for different inputs and chains executions - Be creative!**

In [38]:
chain.run("What caused the fall of the Roman Empire?")



> Entering new MultiPromptChain chain...
History: {'input': 'What caused the fall of the Roman Empire?'}
> Finished chain.


"The fall of the Roman Empire is a complex and debated topic among historians. There are several factors that are believed to have contributed to the decline and eventual fall of the empire. Some of the key factors include:\n\n1. Political instability: The Roman Empire faced frequent changes in leadership, with emperors often being assassinated or overthrown. This instability weakened the central government and made it difficult to effectively govern the vast empire.\n\n2. Economic troubles: The Roman economy struggled with issues such as inflation, high taxes, and a reliance on slave labor. The empire also faced challenges in maintaining trade routes and acquiring new sources of wealth.\n\n3. Military defeats: The Roman Empire faced increasing pressure from barbarian invasions, particularly from Germanic tribes such as the Visigoths and Vandals. The empire's military was stretched thin and unable to effectively defend its borders.\n\n4. Social and cultural decline: The Roman Empire fa

In [39]:
chain.run("What is the difference between a stack and a queue?")



> Entering new MultiPromptChain chain...
computer science: {'input': 'What is the difference between a stack and a queue?'}
> Finished chain.


'A stack is a data structure that follows the Last In, First Out (LIFO) principle, meaning that the last element added to the stack is the first one to be removed. This is similar to a stack of plates where you can only remove the top plate.\n\nOn the other hand, a queue is a data structure that follows the First In, First Out (FIFO) principle, meaning that the first element added to the queue is the first one to be removed. This is similar to a line of people waiting for a bus where the person who arrived first is the first one to board the bus.\n\nIn summary, the main difference between a stack and a queue is the order in which elements are added and removed.'

In [40]:
chain.run("How does CRISPR gene editing work?")



> Entering new MultiPromptChain chain...
biology: {'input': 'How does CRISPR gene editing work?'}
> Finished chain.


"CRISPR gene editing is a revolutionary technology that allows scientists to precisely modify the DNA of living organisms. The CRISPR system is based on a naturally occurring defense mechanism found in bacteria, which uses RNA molecules to target and cut specific sequences of DNA.\n\nIn the lab, researchers can design a guide RNA that matches the target gene they want to edit. This guide RNA is then paired with a protein called Cas9, which acts as a molecular scissors. The guide RNA directs the Cas9 protein to the specific location in the genome, where it cuts the DNA. \n\nOnce the DNA is cut, the cell's natural repair mechanisms come into play. There are two main repair pathways: non-homologous end joining (NHEJ) and homology-directed repair (HDR). NHEJ is error-prone and can introduce small insertions or deletions, leading to gene knockout. HDR, on the other hand, can be used to introduce specific changes or insertions into the DNA.\n\nBy harnessing the power of CRISPR gene editing, 

In [41]:
chain.run("What is the meaning of life?")



> Entering new MultiPromptChain chain...
None: {'input': 'What is the meaning of life?'}
> Finished chain.


'The meaning of life is a deeply philosophical question that has been debated by scholars, theologians, and individuals for centuries. Different people and cultures have different beliefs and interpretations about the purpose and meaning of life. Some believe that the meaning of life is to seek happiness and fulfillment, others believe it is to serve a higher power or fulfill a specific destiny, while others believe that life has no inherent meaning and it is up to each individual to create their own purpose. Ultimately, the meaning of life is a deeply personal and subjective question that each person must grapple with and find their own answer to.'